# Stockwise — 02: Train Models (Colab)

Thin orchestration notebook per CLAUDE.md Section 9. All logic lives in `src/models/`, covered by `tests/test_baselines.py` and `tests/test_metrics.py` (there is no dedicated pytest file for `train_point.py` in the Section 12 manifest -- it's verified by the real comparison this notebook runs, same as `tests/test_leakage.py`'s real-data check in notebook 01).

**Before running:** push any local changes first (`git push`), since Cell 1 clones/pulls from GitHub. This notebook loads the full 10-store feature set from the Drive checkpoint written by `notebooks/01_build_features.ipynb`'s Cell 5 -- run that notebook first if `/content/drive/MyDrive/m5/full/` doesn't exist yet.

**Memory note:** the 10 per-store parquets sum to roughly 15GB combined at their saved dtypes (~1.5GB each), which alone exceeds Colab's ~12.7GB free-tier ceiling. Cell 2 downcasts float64 feature columns to float32 on load, before concatenating, to actually fit.

## Cell 1 — clone or pull, then install

In [ ]:
import os
if not os.path.exists('retail-demand-forecasting'):
    !git clone https://github.com/Kewal-07/retail-demand-forecasting
    %cd retail-demand-forecasting
else:
    %cd retail-demand-forecasting
    !git pull

# Colab already ships pandas/numpy/pyarrow; lightgbm is the only thing
# actually missing. Installing requirements.txt's pins here would force
# the same numpy/pandas downgrade that broke notebook 01's Cell 1.
!pip install -q lightgbm

## Cell 2 — load the full 10-store dataset from Drive, downcasting on the way in

In [ ]:
import glob, gc
import pandas as pd
from google.colab import drive

drive.mount('/content/drive')

store_files = sorted(glob.glob('/content/drive/MyDrive/m5/full/store_id=*/part.parquet'))
print(f"found {len(store_files)} store files")
assert len(store_files) == 10, "expected all 10 stores from notebook 01's Cell 5"

frames = []
for f in store_files:
    part = pd.read_parquet(f)
    for col in part.columns:
        if part[col].dtype == "float64":
            part[col] = part[col].astype("float32")
    frames.append(part)
    print(f, part.shape, part.memory_usage(deep=True).sum() / 1e9, "GB")

df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()
print("combined:", df.shape, df.memory_usage(deep=True).sum() / 1e9, "GB")

## Cell 3 — train/validation split

Filters on the parsed day number, not a sort by `d` -- `d` is category dtype ordered lexicographically (`d_1, d_10, d_11, ..., d_2, ...`), not chronologically. Filtering by a numeric comparison after parsing is safe; sorting by the raw column is not (this is exactly what produced the false-positive leakage check in notebook 01's Cell 6).

In [ ]:
df['day_num'] = df['d'].astype(str).str.replace('d_', '', regex=False).astype(int)

TRAIN_END = 1885
VAL_START, VAL_END = 1886, 1913

train_df = df[df['day_num'] <= TRAIN_END].copy()
val_df = df[(df['day_num'] >= VAL_START) & (df['day_num'] <= VAL_END)].copy()
del df
gc.collect()
print(f"train rows: {len(train_df)}, val rows: {len(val_df)}")

## Cell 4 — global vs per-store comparison

Trains 11 boosters total (1 global + 1 per store) with the same hyperparameters verified locally on CA_1 (point model WRMSSE 0.33 vs seasonal_naive's 1.06). This is the slow cell -- expect it to take a while given the full 59M-row scale.

In [ ]:
from src.models.train_point import compare_global_vs_store

params = {
    "num_leaves": 63,
    "learning_rate": 0.05,
    "min_data_in_leaf": 50,
    "tweedie_variance_power": 1.1,
}

result = compare_global_vs_store(train_df, val_df, params)
print(result)